# Pilot Run — 10 samples per language per task
Run this BEFORE the full inference to verify everything works end-to-end.

In [ ]:
# Install dependencies
!pip install datasets transformers bitsandbytes accelerate openai pandas scipy matplotlib seaborn tqdm -q

In [ ]:
# Step 1: Load datasets
from datasets import load_dataset

tqa_en = load_dataset('alexandrainst/m_truthfulqa', 'en')
tqa_es = load_dataset('alexandrainst/m_truthfulqa', 'es')
xcopa_en = load_dataset('xcopa', 'en')
xcopa_es = load_dataset('xcopa', 'es')
xcopa_sw = load_dataset('xcopa', 'sw')

print('TruthfulQA EN:', len(tqa_en['train']), 'examples')
print('TruthfulQA ES:', len(tqa_es['train']), 'examples')
print('XCOPA EN:', len(xcopa_en['test']), 'examples')
print('XCOPA ES:', len(xcopa_es['test']), 'examples')
print('XCOPA SW:', len(xcopa_sw['test']), 'examples')

In [ ]:
# Step 2: Peek at one example from each
print('=== TruthfulQA EN sample ===')
print(tqa_en['train'][0])
print()
print('=== XCOPA SW sample ===')
print(xcopa_sw['test'][0])

In [ ]:
# Step 3: Load Aya Expanse 8B
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'CohereLabs/aya-expanse-8b'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()
print('Model loaded!')

In [ ]:
# Step 4: Test one prompt
test_prompt = 'Answer truthfully.\nQuestion: What is the largest animal to have ever lived?\nAnswer:'

messages = [{'role': 'user', 'content': test_prompt}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)

new_tokens = outputs[0][inputs.shape[-1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)
print('Response:', response)
print('Token count:', len(new_tokens))